# Context Enrichment - Window Around Chunk

Retrieve the best chunk, then add its neighbours so the LLM sees surrounding context.

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
def retrieve_with_window(query, k=2, window=1):
    hits = store.search(query, k)
    picked = set()
    for h in hits:
        picked.update(range(max(0, h["idx"] - window), min(len(chunks), h["idx"] + window + 1)))
    return [chunks[i] for i in sorted(picked)]   # keep original document order

query = "How does semantic chunking decide where to split?"
narrow = [h["text"] for h in store.search(query, 2)]
wide = retrieve_with_window(query, k=2, window=1)
print(f"{len(narrow)} chunks without window, {len(wide)} chunks with window")

In [ ]:
print("Without window:\n", answer(query, narrow), "\n")
print("With window:\n", answer(query, wide))

Because chunks overlap, adjacent chunks repeat text. For production, merge overlapping spans before prompting.